<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Automatic_differentiation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tensors and Gradient Calculation:Requires grad


In [2]:
import torch
# Define tensors: x (input), w (weight), b (bias)
x = torch.tensor([1.0, 2.0]) # Input data, gradients not needed
w = torch.tensor([0.5, -1.0], requires_grad=True) # Weight parameter, track gradients
b = torch.tensor([0.1], requires_grad=True) # Bias parameter, track gradients

print(f"x requires_grad: {x.requires_grad}")
print(f"w requires_grad: {w.requires_grad}")
print(f"b requires_grad: {b.requires_grad}")

# Perform an operation: y = w * x + b
# Note: PyTorch handles broadcasting for b
intermediate = w * x
print(f"\nintermediate (w * x) requires_grad: {intermediate.requires_grad}")

y = intermediate + b
print(f"y requires_grad: {y.requires_grad}")


x requires_grad: False
w requires_grad: True
b requires_grad: True

intermediate (w * x) requires_grad: True
y requires_grad: True


In [3]:
# Attempting requires_grad on an integer tensor
try:
    int_tensor = torch.tensor([1, 2], dtype=torch.int64, requires_grad=True)
    # This line might not error immediately, but subsequent backward() calls involving it would.
    print(f"Integer tensor created with requires_grad=True: {int_tensor.requires_grad}")
    # Let's try a simple operation that might lead to issues later
    result = int_tensor * 2.0 # Multiply by float to see if it causes issues
    print(f"Result requires_grad: {result.requires_grad}")
    # result.backward() # This would likely fail if we tried to backpropagate
except RuntimeError as e:
    print(f"\nError setting requires_grad on integer tensor: {e}")

# Best practice: Use float tensors for parameters/computations needing gradients
float_tensor = torch.tensor([1.0, 2.0], requires_grad=True)
print(f"\nFloat tensor created with requires_grad=True: {float_tensor.requires_grad}")



Error setting requires_grad on integer tensor: Only Tensors of floating point and complex dtype can require gradients

Float tensor created with requires_grad=True: True


In [4]:
print(f"\nx.grad_fn: {x.grad_fn}")
print(f"w.grad_fn: {w.grad_fn}")
print(f"b.grad_fn: {b.grad_fn}")
print(f"intermediate.grad_fn: {intermediate.grad_fn}") # Result of multiplication
print(f"y.grad_fn: {y.grad_fn}") # Result of addition



x.grad_fn: None
w.grad_fn: None
b.grad_fn: None
intermediate.grad_fn: <MulBackward0 object at 0x7f8c2ff4aef0>
y.grad_fn: <AddBackward0 object at 0x7f8c2ff4aef0>


# Performing Backporpagation:Backward()

In [5]:
import torch

# Example setup (imagine these are results from a model)
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# Perform some operations (building the graph)
y = w * x + b  # y = 3*2 + 1 = 7
loss = y * y   # loss = 7*7 = 49 (a scalar)

# Before backward pass, gradients are None
print(f"Gradient for x before backward: {x.grad}")
print(f"Gradient for w before backward: {w.grad}")
print(f"Gradient for b before backward: {b.grad}")

# Compute gradients
loss.backward()

# After backward pass, gradients are populated
print(f"Gradient for x after backward: {x.grad}") # d(loss)/dx = d(y^2)/dx = 2*y*(dy/dx) = 2*y*w = 2*7*3 = 42
print(f"Gradient for w after backward: {w.grad}") # d(loss)/dw = d(y^2)/dw = 2*y*(dy/dw) = 2*y*x = 2*7*2 = 28
print(f"Gradient for b after backward: {b.grad}") # d(loss)/db = d(y^2)/db = 2*y*(dy/db) = 2*y*1 = 2*7*1 = 14


Gradient for x before backward: None
Gradient for w before backward: None
Gradient for b before backward: None
Gradient for x after backward: 42.0
Gradient for w after backward: 28.0
Gradient for b after backward: 14.0


In [6]:
# Continuing the previous example, but with a non-scalar y
x_vector = torch.tensor([2.0, 4.0], requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y_non_scalar = w * x_vector + b # y_non_scalar is now a non-scalar tensor with two elements: [7.0, 13.0]

try:
    y_non_scalar.backward() # This will cause an error
except RuntimeError as e:
    print(f"Error calling backward() on non-scalar: {e}")

# To make it work, provide a gradient tensor matching y_non_scalar's shape
# This represents the gradient of some final loss w.r.t y.
# For demonstration, let's use torch.ones_like(y_non_scalar)
grad_tensor = torch.ones_like(y_non_scalar)
y_non_scalar.backward(gradient=grad_tensor)
print(f"Gradient for x_vector after y_non_scalar.backward(gradient=...): {x_vector.grad}")
print(f"Gradient for w after y_non_scalar.backward(gradient=...): {w.grad}")


Error calling backward() on non-scalar: grad can be implicitly created only for scalar outputs
Gradient for x_vector after y_non_scalar.backward(gradient=...): tensor([3., 3.])
Gradient for w after y_non_scalar.backward(gradient=...): 6.0


In [12]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

y_non_scalar = w * x_vector + b # y_non_scalar is now a non-scalar tensor with two elements: [7.0, 13.0]
print(f"x_vector shape: {x.shape}")
print(f"w shape: {w.shape}")
print(f"b shape: {b.shape}")
try:
    y_non_scalar.backward()
    print(y_non_scalar) # This will cause an error
except RuntimeError as e:
    print(f"Error calling backward() on non-scalar: {e}")

x_vector shape: torch.Size([])
w shape: torch.Size([1])
b shape: torch.Size([1])
tensor([5.], grad_fn=<AddBackward0>)


# Accessing Gradients

In [13]:
import torch

# Create input tensors that require gradients
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# Define a simple computation
y = w * x + b  # y = 3.0 * 2.0 + 1.0 = 7.0

# Compute gradients
y.backward()

# Access the gradients stored in the .grad attribute
print(f"Gradient of y with respect to x (dy/dx): {x.grad}")
print(f"Gradient of y with respect to w (dy/dw): {w.grad}")
print(f"Gradient of y with respect to b (dy/db): {b.grad}")

# Create a tensor that does NOT require gradients
z = torch.tensor(4.0, requires_grad=False)
print(f"Gradient for tensor z (requires_grad=False): {z.grad}")


Gradient of y with respect to x (dy/dx): 3.0
Gradient of y with respect to w (dy/dw): 2.0
Gradient of y with respect to b (dy/db): 1.0
Gradient for tensor z (requires_grad=False): None


# Disabling Gradient Tracking

In [18]:
import torch

# Example tensors
x = torch.randn(2, 2, requires_grad=True)
w = torch.randn(2, 2, requires_grad=True)
b = torch.randn(2, 2, requires_grad=True)

# Operation outside the no_grad context
y = x * w + b
print(f"y.requires_grad: {y.requires_grad}") # Output: y.requires_grad: True
print(f"y.grad_fn: {y.grad_fn}")
print(f"y.grad_fn.next_functions: {y.grad_fn.next_functions}")          # Output: y.grad_fn: <AddBackward0 object at ...>

# Perform operations within the no_grad context
print("\nEntering torch.no_grad() context:")
with torch.no_grad():
    z = x * w + b
    print(f"  z.requires_grad: {z.requires_grad}") # Output: z.requires_grad: False
    print(f"  z.grad_fn: {z.grad_fn}")           # Output: z.grad_fn: None

    # Even if an input requires grad, the output won't
    k = x * 5
    print(f"  k.requires_grad: {k.requires_grad}") # Output: k.requires_grad: False

# Outside the context, tracking resumes if inputs require grad
print("\nExiting torch.no_grad() context:")
p = x * w
print(f"p.requires_grad: {p.requires_grad}") # Output: p.requires_grad: True
print(f"p.grad_fn: {p.grad_fn}")           # Output: p.grad_fn: <MulBackward0 object at ...>


y.requires_grad: True
y.grad_fn: <AddBackward0 object at 0x7f8c41e52d70>
y.grad_fn.next_functions: ((<MulBackward0 object at 0x7f8c2c35d2a0>, 0), (<AccumulateGrad object at 0x7f8c1168e830>, 0))

Entering torch.no_grad() context:
  z.requires_grad: False
  z.grad_fn: None
  k.requires_grad: False

Exiting torch.no_grad() context:
p.requires_grad: True
p.grad_fn: <MulBackward0 object at 0x7f8c2c35d2a0>


In [19]:
import torch

# Original tensor requiring gradients
a = torch.randn(3, 3, requires_grad=True)
b = a * 2
print(f"b.requires_grad: {b.requires_grad}") # Output: b.requires_grad: True
print(f"b.grad_fn: {b.grad_fn}")           # Output: b.grad_fn: <MulBackward0 object at ...>

# Detach the tensor 'b'
c = b.detach()
print(f"\nAfter detaching b to create c:")
print(f"c.requires_grad: {c.requires_grad}") # Output: c.requires_grad: False
print(f"c.grad_fn: {c.grad_fn}")           # Output: c.grad_fn: None

# Importantly, the original tensor 'b' is unchanged
print(f"\nOriginal tensor b remains attached:")
print(f"b.requires_grad: {b.requires_grad}") # Output: b.requires_grad: True
print(f"b.grad_fn: {b.grad_fn}")           # Output: b.grad_fn: <MulBackward0 object at ...>

# Operations using the detached tensor 'c' won't be tracked
d = c + 1
print(f"\nOperation on detached tensor c:")
print(f"d.requires_grad: {d.requires_grad}") # Output: d.requires_grad: False


b.requires_grad: True
b.grad_fn: <MulBackward0 object at 0x7f8c2ff4a500>

After detaching b to create c:
c.requires_grad: False
c.grad_fn: None

Original tensor b remains attached:
b.requires_grad: True
b.grad_fn: <MulBackward0 object at 0x7f8c2ff4a500>

Operation on detached tensor c:
d.requires_grad: False


In [20]:
my_tensor = torch.randn(5, requires_grad=True)
print(f"Initial requires_grad: {my_tensor.requires_grad}") # Output: Initial requires_grad: True

# Disable gradient tracking in-place
my_tensor.requires_grad_(False) # Note the underscore for in-place operation
print(f"After requires_grad_(False): {my_tensor.requires_grad}") # Output: After requires_grad_(False): False


Initial requires_grad: True
After requires_grad_(False): False


# Gradient Accumulation

In [21]:
import torch

# Create a tensor that requires gradients
x = torch.tensor([2.0], requires_grad=True)

# Perform some operations
y = x * x
z = y * 3 # z = 3 * x^2

# First backward pass
# dz/dx = 6*x = 6*2 = 12
z.backward(retain_graph=True) # retain_graph=True allows subsequent backward calls
print(f"After first backward pass, x.grad: {x.grad}")

# Perform another operation (can be the same or different)
# For simplicity, let's use the same z again for demonstration
# Note: In a real scenario, you'd likely compute a new loss
# based on a new input or different part of the model.
z.backward() # Second backward pass
# We expect the new gradient (12) to be added to the existing one (12)
print(f"After second backward pass, x.grad: {x.grad}")

# Manually zero the gradient
x.grad.zero_()
print(f"After zeroing, x.grad: {x.grad}")


After first backward pass, x.grad: tensor([12.])
After second backward pass, x.grad: tensor([24.])
After zeroing, x.grad: tensor([0.])
